# CLEVER retrieval + repair on Colab (Lean + Mathlib + Ollama)

**First:** Runtime -> Change runtime type -> **T4 GPU**. Then run the cells top to bottom.

Logs, the LLM reply cache and results are kept in Google Drive (`MyDrive/clever-rr/`), so if
Colab disconnects, just run cells 1-5 again and re-run the experiment cell: finished problems are skipped.
Setup (cells 1-5) takes about 20-40 minutes on a fresh runtime.

## 1. Get the code, keep outputs in Drive

In [2]:
import os
from google.colab import drive
drive.mount('/content/drive')

!git clone -q https://github.com/Aditi-Nandiraju/Clever.git /content/Clever || git -C /content/Clever pull -q
%cd /content/Clever

SAVE = '/content/drive/MyDrive/clever-rr'
for d in ['logs', 'cache', 'results']:
    os.makedirs(f'{SAVE}/{d}', exist_ok=True)
    if not os.path.islink(d):
        !rm -rf {d}
        os.symlink(f'{SAVE}/{d}', d)
!ls -l

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
fatal: destination path '/content/Clever' already exists and is not an empty directory.
/content/Clever
total 108
-rw-r--r-- 1 root root 9819 Nov 20 09:26 analyze.py
lrwxrwxrwx 1 root root   38 Nov 20 09:26 cache -> /content/drive/MyDrive/clever-rr/cache
-rw-r--r-- 1 root root 3819 Nov 20 09:26 checker.py
-rw-r--r-- 1 root root 5296 Nov 20 09:26 clever_colab.ipynb
-rw-r--r-- 1 root root 3515 Nov 20 09:26 config.py
-rw-r--r-- 1 root root 8430 Nov 20 09:26 corpus.py
drwxr-xr-x 2 root root 4096 Nov 20 09:26 data
-rw-r--r-- 1 root root 4649 Nov 20 09:26 data.py
-rw-r--r-- 1 root root 2017 Nov 20 09:26 errors.py
-rw-r--r-- 1 root root 4784 Nov 20 09:26 llm.py
lrwxrwxrwx 1 root root   37 Nov 20 09:26 logs -> /content/drive/MyDrive/clever-rr/logs
-rw-r--r-- 1 root root 4433 Nov 20 09:26 prompts.py
-rw-r--r-- 1 root root 7928 Nov 20 09:26 README.md
-rw-r--r-- 1 root 

## 2. Python packages

In [4]:
!pip install -q -r requirements.txt

## 3. Lean 4 + Mathlib
Installs elan, then the Lean version pinned by clever-bench (v4.24.0), downloads prebuilt
Mathlib and builds the CLEVER project. (`clever-bench-install` is not used: it does not
put Lean on the PATH inside Colab.)

In [5]:
import os
!curl -sSfL https://raw.githubusercontent.com/leanprover/elan/master/elan-init.sh | sh -s -- -y --default-toolchain none
os.environ['PATH'] = os.path.expanduser('~/.elan/bin') + ':' + os.environ['PATH']

from clever_bench.benchmark import get_clever_lean_project_path
LEAN_DIR = str(get_clever_lean_project_path())
!cd {LEAN_DIR} && lake exe cache get && lake build 2>&1 | tail -n 5
!cd {LEAN_DIR} && lean --version

info: downloading installer
info: downloading https://releases.lean-lang.org/lean4/v4.24.0/lean-4.24.0-linux.tar.zst
438.3 MiB / 438.3 MiB (100 %) 266.1 MiB/s ETA:   0 s
info: installing /root/.elan/toolchains/leanprover--lean4---v4.24.0
info: mathlib: cloning https://github.com/leanprover-community/mathlib4
info: mathlib: checking out revision 'f897ebcf72cd16f89ab4577d0c826cd14afaafc7'
info: plausible: cloning https://github.com/leanprover-community/plausible
info: plausible: checking out revision 'dfd06ebfe8d0e8fa7faba9cb5e5a2e74e7bd2805'
info: LeanSearchClient: cloning https://github.com/leanprover-community/LeanSearchClient
info: LeanSearchClient: checking out revision '99657ad92e23804e279f77ea6dbdeebaa1317b98'
info: importGraph: cloning https://github.com/leanprover-community/import-graph
info: importGraph: checking out revision 'd768126816be17600904726ca7976b185786e6b9'
info: proofwidgets: cloning https://github.com/leanprover-community/ProofWidgets4
info: proofwidgets: checking 

## 4. Ollama on the GPU
Same model as on the laptop: `qwen2.5:7b` with a 32k context (`qwen2.5-32k`, see `config.py`).

In [ ]:
import subprocess, time
!apt-get -qq install -y zstd > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh
subprocess.Popen(['ollama', 'serve'], stdout=open('/content/ollama.log', 'w'), stderr=subprocess.STDOUT)
time.sleep(5)
open('/content/Modelfile', 'w').write('FROM qwen2.5:7b\nPARAMETER num_ctx 32768\n')
!ollama pull qwen2.5:7b && ollama create qwen2.5-32k -f /content/Modelfile
!ollama list

>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


## 5. Retrieval corpus + offline check

In [ ]:
!python corpus.py
!python smoke_test.py | tail -n 3
!python retriever.py 3

## 6. Real smoke test: 3 dev problems, real model + real Lean
Read a few records in the log before going further.

In [ ]:
!python run_experiment.py --configs C1_baseline --split dev --limit 3 --seeds 0
!head -c 3000 logs/C1_baseline_s0_dev.jsonl

## 7. Dev runs (tune prompts / error rules here, never on test)

In [ ]:
!python run_experiment.py --split dev --seeds 0
!python analyze.py --split dev

## 8. Test runs (reported numbers). Freeze `config.py` and `prompts.py` first.
Long: re-run this cell after a disconnect, it resumes. Results land in Drive: `MyDrive/clever-rr/results/test/`.

In [ ]:
!python run_experiment.py --configs C1_baseline C2_repair
!python run_experiment.py --configs C3_retrieval C4_both
!python analyze.py